# XPS — fitting & publication figures (batches)

Same pattern as every other technique: **`xps.yaml`** holds the shared rules (peak models, background, figure style) and every **batch** (Batch1, Batch2, … — repeat experiments tracked separately).

**Fit or plot is decided per file, automatically:**
* raw Avantage workbook / 2-column spectrum → **fitted here**
* already-fitted export (XPSPeak-style: BE, raw, sum, background, peaks…), yours or a collaborator's → **plotted as-is**, peaks named & coloured from the YAML, so figures look identical
* per batch `mode: plot` never fits, `mode: fit` refits everything

**Outputs per batch** (`<batch>` in every name, nothing ever mixes):
* `Figures/XPS/<batch>/<sample>/<sample>_<region>_<batch>.svg`, `…_Survey_<batch>.svg`, `Survey_stack_<batch>.svg`
* `Exports/PlotData/XPS/<batch>/…` — **each figure's exact data and style, for Origin / SigmaPlot** (same folders as the figures)
* `Exports/XPS_<batch>.xlsx` — area %, atomic %, peak table (± Monte Carlo), fit quality

New batch: drop its files into `Data/XPS/Batch2/` (auto-discovered) or add a `batches:` entry.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import pandas as pd
from battery_analysis.project import Project
from battery_analysis.analysis.xps import fitting as xf, sources, pubplot
from battery_analysis.analysis.xps.tuner import tune

project = Project("..")
project.use("xps", "xps.yaml")

BATCH = "Batch1"                     # the batch this notebook works on
print("XPS batches:", project.xps_batches())

## 0. Tuner (optional) — adjust peaks by eye, then *Save to YAML*
Needs an Avantage workbook in this batch's source. Change `region=`; after saving, re-run from section 1.

In [ ]:
cfg = project.xps_config(BATCH)
wb = sources.survey_workbook(cfg, region="F1s")   # first workbook with this region
tune(wb, project.yaml_dir / "xps.yaml", region="F1s") if wb else print("No Avantage workbook in this batch — tuner skipped.")

## 1. Fit / load the batch (writes the Excel + Origin/SigmaPlot exports)

In [ ]:
results = project.fit_xps(batch=BATCH)
cfg = project.xps_config(BATCH)
xf.quality_table(results)

## 2. Area % within each region (± = Monte Carlo 1σ)

In [ ]:
peaks = xf.results_table(results)
if peaks.empty:
    print("Nothing fitted in this batch.")
else:
    err = peaks["± area %"].astype(str) if "± area %" in peaks else "—"
    display(peaks.assign(**{"area %": peaks["area %"].round(1).astype(str) + " ± " + err})
                 .pivot_table(index=["region", "component"], columns="sample", values="area %",
                              aggfunc="first", sort=False))

## 3. Atomic %

In [ ]:
wb = sources.survey_workbook(cfg)
display(xf.atomic_percent(results, cfg, wb, by="element"))
display(xf.atomic_percent(results, cfg, wb, by="species"))

## 4. Full peak table

In [ ]:
peaks.drop(columns="area", errors="ignore").round(3)

## 5. Review grid (on screen only — judge the fits; not for publication)

In [ ]:
for region in dict.fromkeys(o.region for o in results):
    xf.plot_region(results, region)
    plt.show()

## 6. Publication figures → `Figures/XPS/<batch>/` (their data for Origin / SigmaPlot: `Exports/PlotData/XPS/<batch>/`)

In [ ]:
written = project.plot_xps(batch=BATCH)

In [ ]:
# preview the first sample's set inline (exactly as exported)
first = results[0].sample if results else None
for o in results:
    if o.sample == first:
        pubplot.plot_fit(o, cfg)
        plt.show()

## 7. Every batch at once (same as the other techniques)

In [ ]:
# project.run_all(techniques=["xps"])      # fits + figures + exports for every XPS batch